In [1]:
import pandas as pd
df = pd.read_csv("../enriched_ecommerce_data.csv")

In [2]:
translation = pd.read_csv("../product_category_name_translation.csv")
translation.columns = ["product_category_name", "product_category_name_english"]

df = df.merge(translation, on="product_category_name", how="left")
df['product_category_name_english'] = df['product_category_name_english'].fillna("unknown")

In [11]:
df['text'] = (
    df['product_category_name_english'].astype(str) + " " +
    df['payment_type'].astype(str)
).str.replace("_", " ").str.lower()

In [12]:
df['text'].head()

0    housewares credit card
1        housewares voucher
2        housewares voucher
3          perfumery boleto
4          auto credit card
Name: text, dtype: object

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['text'])

In [14]:
print(vectorizer.get_feature_names_out()[:30])

['accessories' 'agro' 'air' 'and' 'appliances' 'art' 'arts' 'audio' 'auto'
 'baby' 'bags' 'bath' 'beach' 'beauty' 'bed' 'bedroom' 'blu' 'boleto'
 'books' 'business' 'card' 'cds' 'childrens' 'christmas' 'cine' 'clothes'
 'clothing' 'coffee' 'comfort' 'commerce']


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query):
    query = query.lower()
    query_vec = vectorizer.transform([query])
    
    similarity = cosine_similarity(query_vec, X).flatten()

    top_indices = similarity.argsort()[-5:][::-1]

    results = df.iloc[top_indices][[
        'product_category_name_english',
        'payment_type'
    ]].copy()

    results['score'] = similarity[top_indices]

    return results

In [16]:
search("electronics")
search("beauty")


,product_category_name_english,payment_type,score
70717,health_beauty,credit_card,0.66254
51895,health_beauty,credit_card,0.66254
42112,health_beauty,credit_card,0.66254
5682,health_beauty,credit_card,0.66254
72432,health_beauty,credit_card,0.66254


In [ ]:
search("health")
search("computers")
search("home")

,product_category_name_english,payment_type,score
7744,home_construction,credit_card,0.678869
15979,home_construction,credit_card,0.678869
45337,home_construction,credit_card,0.678869
36976,home_construction,credit_card,0.678869
48337,home_construction,credit_card,0.678869
